#### config

In [ ]:
from __future__ import annotations

!pip install sentence-transformers faiss-cpu pandas numpy tqdm python-dotenv >> /dev/null

In [2]:
import json, os, pickle, warnings, rich
from dataclasses import dataclass
from typing import Any, Literal
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import numpy.typing as npt
from sentence_transformers import SentenceTransformer, models
import faiss

warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

In [3]:
@dataclass()
class Config:
    data_dir = '/home/ronji/Dev/CarP/esco_dataset/formatted/'
    db_dir = 'db/'
    model_hf = "Seznam/retromae-small-cs"
    HF_TOKEN = os.getenv('HF_TOKEN')

    # embedding configuration
    batch_size = 256  # size of an encoding batch
    normalize_embeddings = True  # enable cosine similarity (inner product)
    index_type = 'flat'  # store and compare all vectors directly, used instead of IFV (index by similarity, for large datasets)


config = Config()
os.makedirs(config.db_dir + 'pickle', exist_ok=True)

In [11]:
# theres not a config file for sentence transformers on huggingface, so manual config is needed to get better accuracy
word_embedding_model = models.Transformer(
    model_name_or_path="Seznam/simcse-dist-mpnet-paracrawl-cs-en",
    max_seq_length=128,
    )

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=True
)

model_sbert = SentenceTransformer(modules=[word_embedding_model, pooling_model])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Record class
record == ESCO entity

In [12]:
EntityType = Literal[
    'occupation',
    'skill',
    'isco_group',  # categories for occupations
    'skill_group'
]


@dataclass
class Label:
    entity_type: EntityType
    esco_uri: str
    id: str
    preferred_label: str
    alt_label: str
    description: str
    code: str
    isco_code: str

    @property
    def embedding_text(self) -> str:
        text = self.preferred_label
        if self.alt_label:
            text = '{} | {}'.format(text, self.alt_label)
        return text

    def to_json(self) -> dict[str, Any]:
        return {
            'entity_type': self.entity_type,
            'esco_uri': self.esco_uri,
            'id': self.id,
            'preferred_label': self.preferred_label,
            'alt_label': self.alt_label,
            'description': self.description,
            'code': self.code,
            'isco_code': self.isco_code
        }


### Data parsing

In [6]:
def clean_altlabels(col_data):
    return [] if not col_data \
        else [item for item in
              (label.strip() for label in col_data.split('\n'))
              if item]

In [7]:
def load_occupation_data(occ_path) -> list[Label]:
    df = pd.read_csv(occ_path).fillna('')
    records = []
    for _,row in df.iterrows():
        alts = clean_altlabels(row.get('ALTLABELS'))
        if not alts: alts.append('') # so that at least one label is added for each row
        for a in alts:
            records.append(Label(
                entity_type='occupation',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label=a,
                description=' '.join(
                    row.get('DESCRIPTION')
                    .replace('\xa0', ' ')
                    .split()
                ),
                code=row.get('CODE'),
                isco_code=row.get('ISCOGROUPCODE')
            ))
    return records


def load_skill_data(skill_path) -> list[Label]:
    df = pd.read_csv(skill_path).fillna('')
    records = []
    for _,row in df.iterrows():
        alts = clean_altlabels(row.get('ALTLABELS'))
        if not alts: alts.append('')
        for a in alts:
            records.append(Label(
                entity_type='skill',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label=a,
                description=' '.join(
                    row.get('DESCRIPTION')
                    .replace('\xa0', ' ')
                    .split()
                ),
                code='',
                isco_code=''
            ))
    return records


def load_isco(isco_path) -> list[Label]:
    df: pd.DataFrame = pd.read_csv(isco_path).fillna('')
    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type='isco_group',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                alt_label='',
                description='',
                code=row.get('CODE'),
                isco_code=''
            )
        )
    return records


def load_skill_groups(sg_path) -> list[Label]:
    df: pd.DataFrame = pd.read_csv(sg_path, dtype=str, keep_default_na=False).fillna('')
    records = []
    for _, row in df.iterrows():
        records.append(
            Label(
                entity_type='skill_group',
                esco_uri=row.get('ESCOURI'),
                id=row.get('ID'),
                preferred_label=row.get('PREFERREDLABEL'),
                code=row.get('CODE'),
                alt_label='',
                description=' '.join(
                    row.get('DESCRIPTION')
                    .replace('\xa0', ' ')
                    .split()
                ),
                isco_code=''
            )
        )
    return records

In [8]:
occupations_path = config.data_dir + 'combinedOccupations_cs.csv'
skills_path = config.data_dir + 'combinedSkills_cs.csv'
iscogroups_path = config.data_dir + 'ISCOGroups_cs.csv'
skillgroups_path = config.data_dir + 'skillGroups_cs.csv'
labels = []

occ_labels = load_occupation_data(occupations_path)
labels.extend(occ_labels)

skill_labels = load_skill_data(skills_path)
labels.extend(skill_labels)

isco_labels = load_isco(iscogroups_path)
labels.extend(isco_labels)

skillgroup_labels = load_skill_groups(skillgroups_path)
labels.extend(skillgroup_labels)

print(f'labels:                 {len(labels)}\n'
      f'occupations:            {len(occ_labels)}\n'
      f'skills:                 {len(skill_labels)}\n'
      f'skill categories:       {len(skillgroup_labels)}\n'
      f'occupation categories:  {len(isco_labels)}'
      )

labels:                 32718
occupations:            10846
skills:                 20613
skill categories:       640
occupation categories:  619


### Create vector db

In [15]:
def create_index(label_source, db_name, model):
    # encode text segments into embeddings
    texts = [l.embedding_text for l in label_source]
    embeddings = model.encode(
        texts,
        batch_size=config.batch_size,
        show_progress_bar=True,
        normalize_embeddings=config.normalize_embeddings,  # enables cosine similarity
        convert_to_numpy=True
    )
    embeddings = embeddings.astype(np.float32) # faiss uses numpy types

    # build a database index file
    d = embeddings.shape[1] # TODO: notes about ifv/flat
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    faiss.write_index(index,f'{config.db_dir}{db_name}.index')

    # map metadata to vectors
    metadata = [l.to_json() for l in label_source]
    with open(f'{config.db_dir}{db_name}_metadata.json','w',encoding='utf-8') as w:
        json.dump(metadata,w,ensure_ascii=False,indent=4)

    # write pickle files for id lookup option
    # TODO: properly configure paths
    id_idx = {label.id: idx for idx, label in enumerate(label_source)}
    with open(f'{config.db_dir}pickle/{db_name}_id-idx.pkl', 'wb') as f:
        pickle.dump(id_idx, f)

In [17]:
create_index(skill_labels,'skill', model_sbert)
create_index(occ_labels,'occupation',model_sbert)
create_index(isco_labels,'isco_group',model_sbert)
create_index(skillgroup_labels,'skill_group',model_sbert)

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

### Interface

In [18]:
def load_db(entity_type: EntityType = 'all'): # entry function to use in calls
    index = faiss.read_index(f'{config.db_dir}{entity_type}.index')
    with open(f'{config.db_dir}{entity_type}_metadata.json', "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return index, metadata

In [21]:
@dataclass
class Result:
    rank: int
    cosine_score: float # higher = more similar
    entity_type: str
    esco_uri: str
    id: str
    label: str
    code: str
    isco_code: str
    description: str


def search(
    query,
    model,
    ent: EntityType = 'all',
    top_n = 10,
):
    index, metadata = load_db(ent)

    query_vector = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    scores: npt.NDArray[np.float32]
    indices: npt.NDArray[np.int64]
    results = []

    scores, indices = index.search(query_vector, top_n)
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue # faiss for empty
        data = metadata[int(idx)]

        results.append(
            Result(
            rank=len(results) + 1,
            cosine_score=float(score),
            entity_type=data['entity_type'],
            esco_uri=data['esco_uri'],
            label=data['preferred_label'],
            id=data['id'],
            code=data.get('code', ''),
            isco_code=data.get('isco_code',''),
            description=data.get('description','')
            )
        )
    return results


def print_results(results):
    for r in results:
        rich.print(r)
        print()

### testing examples

#### all

In [ ]:
q = input('query all types: ')
results = search(
    query=q,
    model=model_sbert,
    top_n=3)
print_results(results)

#### occupations

In [26]:
q_occ = 'brigáda mcdonalds'
results = search(
    query=q_occ,
    model= model_sbert,
    ent='occupation',
    top_n=3)
results_isco = search(
    query=q_occ,
    model=model_sbert,
    ent='isco_group',
    top_n=3)

print_results(results)
print_results(results_isco)

Result(
    rank=1,
    cosine_score=0.522701621055603,
    entity_type='occupation',
    esco_uri='http://data.europa.eu/esco/occupation/4a1125ee-5728-4b1f-b655-7b8eb2e27660',
    id='key_1519',
    label='vedoucí týmu rychlého občerstvení',
    code='5246.2',
    isco_code=5246.0,
    description='Vedoucí týmu rychlého občerstvení řídí provoz v restauraci s rychlou obsluhou.'
)

Result(
    rank=2,
    cosine_score=0.5100600719451904,
    entity_type='occupation',
    esco_uri='http://data.europa.eu/esco/occupation/4a1125ee-5728-4b1f-b655-7b8eb2e27660',
    id='key_1519',
    label='vedoucí týmu rychlého občerstvení',
    code='5246.2',
    isco_code=5246.0,
    description='Vedoucí týmu rychlého občerstvení řídí provoz v restauraci s rychlou obsluhou.'
)

Result(
    rank=3,
    cosine_score=0.5076909065246582,
    entity_type='occupation',
    esco_uri='http://data.europa.eu/esco/occupation/2afdf635-3cdb-4914-ab61-5dd7d7d46792',
    id='key_1155',
    label='vedoucí prodejny pečiva',
    code='1420.4.5',
    isco_code=1420.0,
    description='Vedoucí prodejen pečiva jsou zodpovědní za činnost specializovaných obchodů a jejich zaměstnance.'
)

Result(
    rank=1,
    cosine_score=0.5643426179885864,
    entity_type='isco_group',
    esco_uri='http://data.europa.eu/esco/isco/C5246',
    id='key_377',
    label='Obsluha v zařízeních rychlého občerstvení',
    code=5246,
    isco_code='',
    description=''
)

Result(
    rank=2,
    cosine_score=0.5228805541992188,
    entity_type='isco_group',
    esco_uri='http://data.europa.eu/esco/isco/C9411',
    id='key_602',
    label='Pracovníci pro přípravu rychlého občerstvení',
    code=9411,
    isco_code='',
    description=''
)

Result(
    rank=3,
    cosine_score=0.5142191648483276,
    entity_type='isco_group',
    esco_uri='http://data.europa.eu/esco/isco/C5212',
    id='key_364',
    label='Pouliční prodavači rychlého občerstvení',
    code=5212,
    isco_code='',
    description=''
)

#### skills
due to varying accuracy Its better to use a cut off rate of around 0.6 for skills, since \
even though specific skills are only found correctly in the whole skill database, \
for most generic skills/occupations the higher category of ESCO is sufficient

In [30]:
q_s = 'COBOL-85'
results = search(
    query=q_s,
    model=model_sbert,
    ent='skill',
    top_n=3)
results_groups = search(
    query=q_s,
    model=model_sbert,
    ent='skill_group',
    top_n=3)

print_results(results)
print_results(results_groups)

Result(
    rank=1,
    cosine_score=0.6684623956680298,
    entity_type='skill',
    esco_uri='http://data.europa.eu/esco/skill/def007fa-5fed-4a5f-91a2-b0d7e3db1be1',
    id='key_17523',
    label='COBOL',
    code='',
    isco_code='',
    description='Techniky a zásady vývoje softwaru, jako je analýza, algoritmy, kódování, testování a sestavování 
programovacích modelů, v programovacím jazyce COBOL.'
)

Result(
    rank=2,
    cosine_score=0.6684623956680298,
    entity_type='skill',
    esco_uri='http://data.europa.eu/esco/skill/def007fa-5fed-4a5f-91a2-b0d7e3db1be1',
    id='key_15701',
    label='COBOL',
    code='',
    isco_code='',
    description='Techniky a zásady vývoje softwaru, jako je analýza, algoritmy, kódování, testování a sestavování 
programovacích modelů, v programovacím jazyce COBOL.'
)

Result(
    rank=3,
    cosine_score=0.4133457839488983,
    entity_type='skill',
    esco_uri='http://data.europa.eu/esco/skill/219dd03d-4168-416a-be40-f4becca55130',
    id='key_17523',
    label='Engrade',
    code='',
    isco_code='',
    description='Počítačový program Engrade je platforma pro elektronické učení zaměřená na vytváření, správu, 
organizování, vykazování a poskytování vzdělávacích kurzů nebo školicích programů v oblasti elektronického učení.'
)

Result(
    rank=1,
    cosine_score=0.29706552624702454,
    entity_type='skill_group',
    esco_uri='http://data.europa.eu/esco/isced-f/1014',
    id='key_17721',
    label='sporty',
    code='',
    isco_code='',
    description=''
)

Result(
    rank=2,
    cosine_score=0.2597894072532654,
    entity_type='skill_group',
    esco_uri='http://data.europa.eu/esco/isced-f/0916',
    id='key_17700',
    label='farmacie',
    code='',
    isco_code='',
    description=''
)

Result(
    rank=3,
    cosine_score=0.24739688634872437,
    entity_type='skill_group',
    esco_uri='http://data.europa.eu/esco/isced-f/0531',
    id='key_17624',
    label='chemie',
    code='',
    isco_code='',
    description=''
)